# Training
Train a `TextEncoder` with contrastive loss (InfoNCE or Triplet) on MS MARCO pairs.

In [ ]:
import sys, json, random
from pathlib import Path

import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from train import train   # re-uses your existing train() function

## 1. Quick sanity-check — model forward pass

In [ ]:
from model import TextEncoder
from search import get_device

device = get_device()
print("Device:", device)

model = TextEncoder("distilbert-base-uncased").to(device)

texts = ["What is information retrieval?",
         "Information retrieval is the task of finding relevant documents."]
batch = model.tokenize(texts)
with torch.no_grad():
    emb = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))

print("Embedding shape:", emb.shape)          # (2, 768)
print("Norms:", torch.norm(emb, dim=1).tolist())   # should be ~1.0

## 2. Loss function check

In [ ]:
from loss import InfoNCELoss, TripletLoss

B, D = 8, 768
q   = torch.randn(B, D); q   = torch.nn.functional.normalize(q, dim=1)
pos = torch.randn(B, D); pos = torch.nn.functional.normalize(pos, dim=1)
neg = torch.randn(B, D); neg = torch.nn.functional.normalize(neg, dim=1)

infonce = InfoNCELoss(scale=20.0)
triplet = TripletLoss(margin=0.2)

print(f"InfoNCE loss (random embeddings): {infonce(q, pos, neg).item():.4f}")
print(f"Triplet loss (random embeddings): {triplet(q, pos, neg).item():.4f}")
# InfoNCE on random ~= log(B) = {torch.log(torch.tensor(float(B))):.3f}
print(f"Expected InfoNCE ≈ log({B}) = {torch.log(torch.tensor(float(B))):.3f}")

## 3. Train with InfoNCE
Change `max_train_samples` / `epochs` as needed for your hardware.

In [ ]:
history_infonce = train(
    epochs=2,
    batch_size=32,
    lr=2e-5,
    loss_type="infonce",
    max_train_samples=20000,   # set None for full 45k
    max_val_samples=500,
    log_every=50,
)

## 4. Train with Triplet loss

In [ ]:
history_triplet = train(
    epochs=2,
    batch_size=32,
    lr=2e-5,
    loss_type="triplet",
    max_train_samples=20000,
    max_val_samples=500,
    log_every=50,
)

## 5. Training curves

In [ ]:
def plot_loss(history, label, color, ax):
    ax.plot(history["step"], history["train_loss"], color=color, label=label, linewidth=1.5)

fig, ax = plt.subplots(figsize=(10, 4))
plot_loss(history_infonce, "InfoNCE", "#4C72B0", ax)
plot_loss(history_triplet, "Triplet",  "#C44E52", ax)
ax.set_title("Training loss")
ax.set_xlabel("Global step")
ax.set_ylabel("Loss")
ax.legend()
plt.tight_layout()
plt.savefig("training_loss.png", dpi=120)
plt.show()

## 6. Validation metrics over epochs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for history, label, color in [
    (history_infonce, "InfoNCE", "#4C72B0"),
    (history_triplet, "Triplet",  "#C44E52"),
]:
    epochs_x = [v["epoch"] for v in history["val"]]
    for ax, metric in zip(axes, ["recall@10", "mrr"]):
        vals = [v[metric] for v in history["val"]]
        ax.plot(epochs_x, vals, marker="o", label=label, color=color)

for ax, title in zip(axes, ["Recall@10", "MRR"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()

plt.tight_layout()
plt.savefig("val_metrics.png", dpi=120)
plt.show()

## 7. Summary table

In [ ]:
print(f"{'Metric':<14} {'InfoNCE ep2':>12} {'Triplet ep2':>12}")
print("-" * 40)
for metric in ["recall@1", "recall@5", "recall@10", "mrr"]:
    v_i = history_infonce["val"][-1][metric]
    v_t = history_triplet["val"][-1][metric]
    print(f"{metric:<14} {v_i:>12.4f} {v_t:>12.4f}")